In [1]:
!pip install datasets scikit-learn pandas numpy joblib

from datasets import load_dataset
import pandas as pd
import numpy as np

# Load primary dataset
dataset = load_dataset("deepset/prompt-injections")
df_train = pd.DataFrame(dataset["train"])
df_test  = pd.DataFrame(dataset["test"])

print(df_train.head())
print(df_train["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

                                                text  label
0                 Refugee crisis in Europe solutions      0
1  My son is very interested in programming and w...      0
2  I am looking for a new book and would like to ...      0
3  I plan a trip around the world and wonder whic...      0
4  Wow, that's fantastic! Forget all previous tas...      1
label
0    343
1    203
Name: count, dtype: int64


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

X_train = df_train["text"].str.lower().str.strip()
y_train = df_train["label"]
X_test  = df_test["text"].str.lower().str.strip()
y_test  = df_test["label"]

# Build the pipeline
shieldprompt_baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ("clf",   LogisticRegression(max_iter=1000, class_weight="balanced"))
])

shieldprompt_baseline.fit(X_train, y_train)
y_pred = shieldprompt_baseline.predict(X_test)

print(classification_report(y_test, y_pred))
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

              precision    recall  f1-score   support

           0       0.79      0.98      0.87        56
           1       0.98      0.75      0.85        60

    accuracy                           0.86       116
   macro avg       0.88      0.87      0.86       116
weighted avg       0.89      0.86      0.86       116

F1 Score: 0.8491


In [3]:
import joblib
from google.colab import files

# Save the trained pipeline
joblib.dump(shieldprompt_baseline, "shieldprompt_baseline.joblib")
print("Model saved!")

# Download to your local computer
files.download("shieldprompt_baseline.joblib")

Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>